# Foundations A2: Working Along a Dimension

Companion notebook for [Foundations A2 of the interactive course](https://bc-sparknet.github.io/make-an-llm-2/#/chapter/dimensions). The code cells follow the chapter's examples in order; the explanations are on the site.

*This notebook is generated from the chapter source. To change it, edit the chapter and run `npm run notebooks`.*

In [ ]:
%pip install -q torch tiktoken

## The one rule

In [ ]:
import torch

x = torch.tensor([[[3, 1, 4, 1],
                   [5, 9, 2, 6],
                   [5, 3, 5, 8]],
                  [[9, 7, 9, 3],
                   [2, 3, 8, 4],
                   [6, 2, 6, 4]]])
print(x.shape)  # torch.Size([2, 3, 4])

In [ ]:
print(x.sum(dim=0).shape)  # torch.Size([3, 4])   the 2 is gone
print(x.sum(dim=1).shape)  # torch.Size([2, 4])   the 3 is gone
print(x.sum(dim=2).shape)  # torch.Size([2, 3])   the 4 is gone

In [ ]:
print(x.sum(dim=-1).shape)  # torch.Size([2, 3])   same as dim=2
print(x.sum(dim=-2).shape)  # torch.Size([2, 4])   same as dim=1

## The picture: walk along d and squash it

In [ ]:
print(x.sum(dim=0))
# tensor([[12,  8, 13,  4],
#         [ 7, 12, 10, 10],
#         [11,  5, 11, 12]])

print(x.sum(dim=2))
# tensor([[ 9, 22, 21],
#         [28, 17, 18]])

print(x.float().mean(dim=-1))   # mean needs floats
# tensor([[2.2500, 5.5000, 5.2500],
#         [7.0000, 4.2500, 4.5000]])

In [ ]:
result = x.max(dim=1)            # walk down the 3 tokens
print(result.values)
# tensor([[5, 9, 5, 8],
#         [9, 7, 9, 4]])
print(result.indices)            # which token (0, 1 or 2) held each max
# tensor([[1, 1, 2, 2],
#         [0, 0, 0, 1]])

## keepdim=True: squash, but leave a slot

In [ ]:
xf = x.float()
mu = xf.mean(dim=-1, keepdim=True)
print(mu.shape)  # torch.Size([2, 3, 1])

In [ ]:
centered = xf - mu               # [2, 3, 4] - [2, 3, 1] -> [2, 3, 4]
print(centered[0])
# tensor([[ 0.7500, -1.2500,  1.7500, -1.2500],
#         [-0.5000,  3.5000, -3.5000,  0.5000],
#         [-0.2500, -2.2500, -0.2500,  2.7500]])
print(centered.mean(dim=-1))     # every token vector now has mean 0
# tensor([[0., 0., 0.],
#         [0., 0., 0.]])

In [ ]:
try:
    xf - xf.mean(dim=-1)         # [2, 3, 4] - [2, 3]
except RuntimeError as e:
    print(e)
# The size of tensor a (4) must match the size of tensor b (3) at non-singleton dimension 2

## Softmax along a dimension

In [ ]:
scores = torch.tensor([[2.0, 1.0, 0.1, 0.5],
                       [0.3, 2.5, 0.2, 1.0],
                       [1.2, 0.4, 3.0, 0.0]])

weights = torch.softmax(scores, dim=-1)
print(weights.round(decimals=2))
# tensor([[0.5700, 0.2100, 0.0900, 0.1300],
#         [0.0800, 0.7000, 0.0700, 0.1600],
#         [0.1300, 0.0600, 0.7800, 0.0400]])
print(weights.sum(dim=-1))   # each row sums to 1
# tensor([1., 1., 1.])

In [ ]:
wrong = torch.softmax(scores, dim=0)
print(wrong.sum(dim=-1))     # rows: not 1
# tensor([1.1351, 1.4161, 1.4488])
print(wrong.sum(dim=0))      # columns: 1
# tensor([1.0000, 1.0000, 1.0000, 1.0000])

## Layer norm, step by step

In [ ]:
h = torch.tensor([[2.0, 4.0, 6.0, 8.0],     # token 0
                  [1.0, 1.0, 1.0, 5.0],     # token 1
                  [10.0, 0.0, 5.0, 5.0]])   # token 2

mu = h.mean(dim=-1, keepdim=True)            # one mean per token: [3, 1]
print(mu)
# tensor([[5.],
#         [2.],
#         [5.]])

centered = h - mu                            # [3, 4] - [3, 1]
print(centered)
# tensor([[-3., -1.,  1.,  3.],
#         [-1., -1., -1.,  3.],
#         [ 5., -5.,  0.,  0.]])

var = centered.pow(2).mean(dim=-1, keepdim=True)   # one variance per token: [3, 1]
print(var)
# tensor([[ 5.0000],
#         [ 3.0000],
#         [12.5000]])

normed = centered / torch.sqrt(var + 1e-5)   # [3, 4] / [3, 1]
print(normed.round(decimals=2))
# tensor([[-1.3400, -0.4500,  0.4500,  1.3400],
#         [-0.5800, -0.5800, -0.5800,  1.7300],
#         [ 1.4100, -1.4100,  0.0000,  0.0000]])

In [ ]:
print(normed.mean(dim=-1).round(decimals=4))                  # tensor([0., 0., 0.])
print(normed.var(dim=-1, unbiased=False).round(decimals=4))   # tensor([1., 1., 1.])

In [ ]:
ln = torch.nn.LayerNorm(4, elementwise_affine=False)  # no learnable scale/shift
print(torch.allclose(ln(h), normed, atol=1e-6))       # True

In [ ]:
wrong = (h - h.mean(dim=0, keepdim=True)) / torch.sqrt(h.var(dim=0, keepdim=True, unbiased=False) + 1e-5)
print(wrong.mean(dim=-1).round(decimals=2))   # token vectors are NOT centred
# tensor([ 0.7800, -0.8300,  0.0500])

## Broadcasting

In [ ]:
a = torch.zeros(3, 4)
b = torch.tensor([10., 20., 30., 40.])
print(a + b)
# tensor([[10., 20., 30., 40.],
#         [10., 20., 30., 40.],
#         [10., 20., 30., 40.]])

In [ ]:
tok = torch.zeros(2, 3, 4)                    # token embeddings: [batch, tokens, emb]
pos = torch.tensor([[1., 1., 1., 1.],         # position 0
                    [2., 2., 2., 2.],         # position 1
                    [3., 3., 3., 3.]])        # position 2   -> [tokens, emb]
out = tok + pos
print(out.shape)                   # torch.Size([2, 3, 4])
print(torch.equal(out[0], out[1])) # True: both texts got the same position vectors

In [ ]:
col = torch.tensor([[1.], [2.], [3.]])        # [3, 1]
row = torch.tensor([[10., 20., 30., 40.]])    # [1, 4]
print(col + row)                              # [3, 4]: an "addition table"
# tensor([[11., 21., 31., 41.],
#         [12., 22., 32., 42.],
#         [13., 23., 33., 43.]])

In [ ]:
try:
    a + torch.tensor([1., 2., 3.])
except RuntimeError as e:
    print(e)
# The size of tensor a (4) must match the size of tensor b (3) at non-singleton dimension 1

In [ ]:
print(torch.broadcast_shapes((2, 3, 4), (3, 4)))     # torch.Size([2, 3, 4])
print(torch.broadcast_shapes((2, 3, 4), (2, 3, 1)))  # torch.Size([2, 3, 4])

## argmax: picking tokens

In [ ]:
logits = torch.tensor([[[0.1, 2.0, -1.0, 0.5, 0.3],
                        [1.5, 0.2,  0.1, 0.0, 2.2],
                        [0.0, 0.0,  3.1, 0.4, 0.2]],
                       [[2.5, 0.1,  0.3, 0.9, 0.0],
                        [0.2, 0.2,  0.2, 1.7, 0.1],
                        [0.3, 4.0,  0.0, 0.0, 0.1]]])
print(logits.shape)            # torch.Size([2, 3, 5])

ids = logits.argmax(dim=-1)
print(ids)
# tensor([[1, 4, 2],
#         [0, 3, 1]])
print(ids.shape)               # torch.Size([2, 3]): one token ID per position

In [ ]:
last = logits[:, -1, :]                        # [2, 5]: last position of each text
next_ids = last.argmax(dim=-1, keepdim=True)   # [2, 1]
print(next_ids)
# tensor([[2],
#         [1]])

probs = torch.softmax(last, dim=-1)            # softmax keeps the order,
print(torch.equal(probs.argmax(dim=-1, keepdim=True), next_ids))  # True: same pick